In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_groq import ChatGroq
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph.message import add_messages
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma,FAISS
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_community.document_loaders import PyPDFLoader
import os
import sqlite3

load_dotenv()

True

In [3]:
llm = ChatGroq(model="llama-3.3-70b-versatile", 
                 api_key=os.getenv("API_KEY"),
                 temperature=0.1)

embeddings = HuggingFaceEndpointEmbeddings(
    model="sentence-transformers/all-mpnet-base-v2",
    huggingfacehub_api_token=os.getenv("HUGGINGFACE_API_KEY")
)

c:\Users\chandu\Desktop\Corrective_RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### PDF Loader

In [4]:
docs=PyPDFLoader("Leave-Policy.pdf").load()

In [5]:
for i in range(len(docs)):
    print(docs[i].page_content)
    print("-"*50)

Leave Policy
General Overview of Leave policy :
Employees, across all industries in Goa, India, are entitled to a certain number of leaves per
year aside from the holidays and days off. In units covered under The Factories Act, three
types of leaves are generally followed namely earned leave, sick leave and casual leave
which an employee can avail without loss of pay.
 Casual leave.
 Sick leave 
 Privilege leave 
Eligibility- 
All regular employees are entitled to leave as per the standard leave policy.
How it works – 
Commencement of Leave Period is calendar year i.e. 1st January to 31st December every
year.   
Employee need to apply for each leave and take approval except in cases where approval
could not be taken in advance usually for casual or sick leaves. 
Grant of leave shall depend upon the policies of the workplace and is at the discretion of the
manager/management. 
There is no set rule for which leave to be approved and not approved.  Employer can refuse
the leave applica

### Text Splitters

In [6]:
chunks = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50).split_documents(docs)

In [7]:
chunks[0].page_content

'Leave Policy\nGeneral Overview of Leave policy :\nEmployees, across all industries in Goa, India, are entitled to a certain number of leaves per\nyear aside from the holidays and days off. In units covered under The Factories Act, three\ntypes of leaves are generally followed namely earned leave, sick leave and casual leave\nwhich an employee can avail without loss of pay.\n\uf0b7 Casual leave.\n\uf0b7 Sick leave \n\uf0b7 Privilege leave \nEligibility-'

### Vector Stores

In [8]:
vectorstore = Chroma.from_documents(chunks, embeddings, persist_directory="basic_RAG/Chroma_db")

In [9]:
vectorstore

In [10]:
vectorstore.get(include=["documents"], ids=["0cab7333-0cc8-4ad7-bee8-e37b207d8fa3"])

{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['documents'],
 'data': None,
 'metadatas': None}

### Retriever

In [11]:
retriever=vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [12]:
retrieved_docs=retriever.invoke("What is the transformer")

In [13]:
for i in range(len(retrieved_docs)):
    print(retrieved_docs[i].page_content)
    print("-"*50)

 Work from home: Employers may allow Eligible Employees to work from home on a
case to case basis depending on the nature of work. The conditions governing such
work from home may be mutually agreed between the employer and the employee
6. Paternity leave :  
Paternity leave is unpaid leave given to a male employee when a child is born, duration of
the leave is at the sole discretion of the management.
7. Leave without pay :
--------------------------------------------------
Privilege leave is provided for planned long leaves for the purpose of travel, vacation etc.
 One  day  PL/EL  leave  is  credited  to  employees  leave  account  for  every  twenty
working  days  provided  an  employee  is  on  continuous  service.  Here  Continuous
service is defined as an employee should work at least for 240 days out of 360 days,
or in case he/she joined in middle of the calendar year it should be 2/3rd of days.
--------------------------------------------------
from the date you start employ

### Generate answer with retrieved docs

In [14]:
val=[doc.page_content for doc in retrieved_docs]
print(val)

['\uf0b7 Work from home: Employers may allow Eligible Employees to work from home on a\ncase to case basis depending on the nature of work. The conditions governing such\nwork from home may be mutually agreed between the employer and the employee\n6. Paternity leave :  \nPaternity leave is unpaid leave given to a male employee when a child is born, duration of\nthe leave is at the sole discretion of the management.\n7. Leave without pay :', 'Privilege leave is provided for planned long leaves for the purpose of travel, vacation etc.\n\uf0b7 One  day  PL/EL  leave  is  credited  to  employees  leave  account  for  every  twenty\nworking  days  provided  an  employee  is  on  continuous  service.  Here  Continuous\nservice is defined as an employee should work at least for 240 days out of 360 days,\nor in case he/she joined in middle of the calendar year it should be 2/3rd of days.', 'from the date you start employment through December 31 of that calendar year.']


In [15]:
ans=llm.invoke([doc.page_content for doc in retrieved_docs])

In [16]:
ans.content

"It seems like we're discussing employee leave policies. To summarize:\n\n1. **Work from home**: Eligible employees may be allowed to work from home on a case-by-case basis, with conditions mutually agreed upon by the employer and employee.\n2. **Paternity leave**: Male employees may take unpaid leave at the discretion of management when a child is born.\n3. **Leave without pay**: No details are provided, but it's mentioned as a separate type of leave.\n4. **Privilege leave (PL) / Earned leave (EL)**: \n   - One day of PL/EL is credited to an employee's leave account for every 20 working days.\n   - To be eligible, an employee must be on continuous service, defined as:\n     - Working at least 240 days out of 360 days in a calendar year.\n     - Or, if joining mid-year, working at least 2/3 of the days from the start date to December 31 of that calendar year.\n\nIs there anything specific you'd like to know or discuss regarding these leave policies?"